# Hindi Hostility Detection — Transformer & LLM experiments (Colab)
Runtime → Change runtime type → **T4 GPU**. Run cells top to bottom.
Outputs: `transformer_coarse.csv`, `transformer_fine.csv` and `llm_results.csv`. Send these files back to fill the [RUN COLAB] cells of the paper. Each part saves its CSV after every model, so a disconnect loses only the model in progress. Part A takes about 45–60 min on a T4; Part B about 1.5–2 h (run it in a second session if needed).
Same splits, labels and metrics as the classical experiments (official CONSTRAINT-2021 train/valid/test).

In [ ]:
# Save everything in Google Drive so results survive a disconnect
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/hostility
%cd /content/drive/MyDrive/hostility

In [ ]:
!pip -q install transformers accelerate bitsandbytes scikit-learn pandas
![ -d Hindi-Hostility-Detection-CONSTRAINT-2021 ] || git clone -q --depth 1 https://github.com/mohit19014/Hindi-Hostility-Detection-CONSTRAINT-2021.git
D = 'Hindi-Hostility-Detection-CONSTRAINT-2021/Dataset'

In [ ]:
import numpy as np, pandas as pd, torch, time, json, re, random
from sklearn.metrics import f1_score, accuracy_score
SEED = 42
FINE = ['fake','hate','offensive','defamation']
tr, va, te = (pd.read_csv(f'{D}/{s}.csv') for s in ['train','valid','test'])
def labels(df):
    sets = df['Labels Set'].str.split(',').apply(lambda l: {x.strip() for x in l})
    yc = (~sets.apply(lambda s: 'non-hostile' in s)).astype(int).values
    yf = np.stack([sets.apply(lambda s, l=l: int(l in s)).values for l in FINE], 1)
    return yc, yf
ytr_c, ytr_f = labels(tr); yva_c, yva_f = labels(va); yte_c, yte_f = labels(te)
T = {k: d['Post'].fillna('').astype(str).tolist() for k, d in [('tr',tr),('va',va),('te',te)]}
def score(pc, pf):
    per = [f1_score(yte_f[:,j], pf[:,j]) for j in range(4)]; sup = yte_f.sum(0)
    return dict(coarse_acc=accuracy_score(yte_c,pc), coarse_F1_hostile=f1_score(yte_c,pc),
                coarse_macroF1=f1_score(yte_c,pc,average='macro'), coarse_weightedF1=f1_score(yte_c,pc,average='weighted'),
                F1_fake=per[0], F1_hate=per[1], F1_offensive=per[2], F1_defamation=per[3],
                fine_weightedF1=float(np.dot(per,sup)/sup.sum()))
URL = re.compile(r'https?://\S+|www\.\S+'); MEN = re.compile(r'@\w+'); HASH = re.compile(r'#\S+')
def clean(s): return HASH.sub(' ', MEN.sub(' ', URL.sub(' ', s)))   # removes URLs, @mentions, #hashtags
TEXT = {'raw': T, 'clean': {k: [clean(x) for x in v] for k, v in T.items()}}
print(len(tr), len(va), len(te))

## Part A — Fine-tuned transformers (mBERT, MuRIL, IndicBERT)
IndicBERT is a gated model: first open https://huggingface.co/ai4bharat/indic-bert, log in, and click the button to agree and get access. Then make a token at https://huggingface.co/settings/tokens (type: Read) and paste it when the next cell asks.
Results are saved in two files, `transformer_coarse.csv` and `transformer_fine.csv`. A run that is already in a file is skipped, so after a disconnect just run Part A again.

In [ ]:
from huggingface_hub import login
login()   # paste your Hugging Face Read token

In [ ]:
!pip -q install sentencepiece
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
dev = 'cuda'
MODELS = {'mBERT':'bert-base-multilingual-cased', 'MuRIL':'google/muril-base-cased', 'IndicBERT':'ai4bharat/indic-bert'}
EPOCHS, LR, BS, MAXLEN = 4, 2e-5, 16, 128
GRID = np.arange(0.05, 0.96, 0.05)

def enc(tok, texts):
    e = tok(texts, truncation=True, max_length=MAXLEN, padding='max_length', return_tensors='pt')
    return e['input_ids'], e['attention_mask']

def run(path, y_tr, y_va, multilabel, T):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    tok = AutoTokenizer.from_pretrained(path)
    m = AutoModelForSequenceClassification.from_pretrained(path, num_labels=4 if multilabel else 2).to(dev)
    ids, am = enc(tok, T['tr']); yt = torch.tensor(y_tr, dtype=torch.float if multilabel else torch.long)
    dl = DataLoader(TensorDataset(ids, am, yt), batch_size=BS, shuffle=True)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.1*len(dl)*EPOCHS), len(dl)*EPOCHS)
    # multi-label: weight each label's positives by neg/pos (rare labels such as defamation are not ignored)
    pw = torch.tensor((len(y_tr) - y_tr.sum(0)) / y_tr.sum(0), dtype=torch.float, device=dev) if multilabel else None
    def scores(split):  # probabilities (multi-label) or class predictions (coarse)
        i, a = enc(tok, T[split]); m.eval(); out = []
        with torch.no_grad():
            for b in range(0, len(i), 64):
                lg = m(input_ids=i[b:b+64].to(dev), attention_mask=a[b:b+64].to(dev)).logits
                out.append(torch.sigmoid(lg).cpu() if multilabel else lg.argmax(-1).cpu())
        return torch.cat(out).numpy()
    def tune(pv):  # best threshold per label on validation
        th = [max(GRID, key=lambda t: f1_score(y_va[:,j], (pv[:,j] > t).astype(int))) for j in range(4)]
        return th, np.mean([f1_score(y_va[:,j], (pv[:,j] > th[j]).astype(int)) for j in range(4)])
    best, best_state, best_th = -1, None, None
    t0 = time.perf_counter()
    for ep in range(EPOCHS):
        m.train()
        for i, a, y in dl:
            lg = m(input_ids=i.to(dev), attention_mask=a.to(dev)).logits
            loss = F.binary_cross_entropy_with_logits(lg, y.to(dev), pos_weight=pw) if multilabel else F.cross_entropy(lg, y.to(dev))
            loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sch.step(); opt.zero_grad()
        pv = scores('va')
        if multilabel: th, s = tune(pv)
        else: th, s = None, f1_score(y_va, pv, average='macro')
        print(f'  epoch {ep+1}: validation score {s:.4f}')
        if s > best: best, best_state, best_th = s, {k: v.detach().clone() for k, v in m.state_dict().items()}, th
    train_t = time.perf_counter() - t0
    m.load_state_dict(best_state)
    torch.cuda.synchronize(); t0 = time.perf_counter(); p = scores('te'); torch.cuda.synchronize()
    inf = (time.perf_counter() - t0) / len(T['te']) * 1e6  # ms per 1k posts
    if multilabel: p = np.stack([(p[:,j] > best_th[j]).astype(int) for j in range(4)], 1)
    nparams = sum(x.numel() for x in m.parameters()) / 1e6
    del m; torch.cuda.empty_cache()
    return p, train_t, inf, nparams, best_th

def load(f): return pd.read_csv(f) if os.path.exists(f) else pd.DataFrame()
# keep the coarse results from the first run (they were correct)
if os.path.exists('transformer_results.csv') and not os.path.exists('transformer_coarse.csv'):
    old = pd.read_csv('transformer_results.csv')
    old[['model','text','params_M','train_time_coarse_s','infer_ms_per_1k_GPU','coarse_acc','coarse_F1_hostile','coarse_macroF1','coarse_weightedF1']].to_csv('transformer_coarse.csv', index=False)

for variant in ['raw', 'clean']:
    for name, path in MODELS.items():
        C = load('transformer_coarse.csv')
        if C.empty or not ((C.model == name) & (C.text == variant)).any():
            print(name, variant, 'coarse')
            pc, tt, inf, npar, _ = run(path, ytr_c, yva_c, False, TEXT[variant])
            r = dict(model=name, text=variant, params_M=round(npar,1), train_time_coarse_s=tt, infer_ms_per_1k_GPU=inf,
                     coarse_acc=accuracy_score(yte_c,pc), coarse_F1_hostile=f1_score(yte_c,pc),
                     coarse_macroF1=f1_score(yte_c,pc,average='macro'), coarse_weightedF1=f1_score(yte_c,pc,average='weighted'))
            pd.concat([C, pd.DataFrame([r])]).to_csv('transformer_coarse.csv', index=False); print(r)
        Fn = load('transformer_fine.csv')
        if Fn.empty or not ((Fn.model == name) & (Fn.text == variant)).any():
            print(name, variant, 'fine')
            pf, tt, _, _, th = run(path, ytr_f, yva_f, True, TEXT[variant])
            per = [f1_score(yte_f[:,j], pf[:,j]) for j in range(4)]; sup = yte_f.sum(0)
            r = dict(model=name, text=variant, train_time_fine_s=tt, thresholds=str([round(float(t),2) for t in th]),
                     F1_fake=per[0], F1_hate=per[1], F1_offensive=per[2], F1_defamation=per[3], fine_weightedF1=float(np.dot(per,sup)/sup.sum()))
            pd.concat([Fn, pd.DataFrame([r])]).to_csv('transformer_fine.csv', index=False); print(r)
print(load('transformer_coarse.csv')); print(load('transformer_fine.csv'))

## Part B — Zero-shot and few-shot LLMs (Qwen2.5-1.5B / 7B Instruct, 4-bit)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
LLMS = {'Qwen2.5-1.5B-Instruct':'Qwen/Qwen2.5-1.5B-Instruct', 'Qwen2.5-7B-Instruct':'Qwen/Qwen2.5-7B-Instruct'}
SYS = ('You are a content moderator for Hindi social media posts. Classify the post. '
       'If it is not hostile, answer exactly: non-hostile. Otherwise answer with one or more of '
       'fake, hate, offensive, defamation separated by commas. '
       'fake = false or misleading news; hate = attacks a group by religion, caste, gender etc.; '
       'offensive = abusive, insulting or vulgar language; defamation = false claims damaging a specific person or organisation. '
       'Answer with the labels only.')
# 5 fixed few-shot examples from the training split (one per label type)
rng = np.random.default_rng(SEED); shots = []
for lab in ['non-hostile','fake','hate','offensive','defamation']:
    idx = np.where(tr['Labels Set'].str.strip() == lab)[0]; shots.append(int(rng.choice(idx)))

def parse(txt):
    t = txt.lower()
    f = [int(l in t) for l in FINE]
    return int(any(f)), f  # no recognised hostile label => non-hostile

def run_llm(path, k_shot, T, variant):
    tok = AutoTokenizer.from_pretrained(path); tok.padding_side = 'left'
    m = AutoModelForCausalLM.from_pretrained(path, device_map='auto',
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16))
    def msgs(post):
        ms = [{'role':'system','content':SYS}]
        if k_shot:
            for s in shots:
                ms += [{'role':'user','content':(clean(tr['Post'][s]) if variant == 'clean' else tr['Post'][s])[:600]}, {'role':'assistant','content':tr['Labels Set'][s]}]
        return ms + [{'role':'user','content':post[:600]}]
    prompts = [tok.apply_chat_template(msgs(p), tokenize=False, add_generation_prompt=True) for p in T['te']]
    outs = []; t0 = time.perf_counter()
    for b in range(0, len(prompts), 16):
        e = tok(prompts[b:b+16], return_tensors='pt', padding=True).to('cuda')
        with torch.no_grad():
            g = m.generate(**e, max_new_tokens=15, do_sample=False)
        outs += tok.batch_decode(g[:, e['input_ids'].shape[1]:], skip_special_tokens=True)
    inf = (time.perf_counter() - t0) / len(prompts) * 1e6
    del m; torch.cuda.empty_cache()
    parsed = [parse(o) for o in outs]
    return np.array([p[0] for p in parsed]), np.array([p[1] for p in parsed]), inf, outs

rows = []
for variant in ['raw', 'clean']:
  for name, path in LLMS.items():
    for k in (0, 5):
        pc, pf, inf, raw = run_llm(path, k, TEXT[variant], variant)
        r = dict(model=name, text=variant, setting=f'{k}-shot', infer_ms_per_1k_GPU=inf, **score(pc, pf)); rows.append(r); print(r)
        json.dump(raw, open(f'raw_{name}_{k}shot_{variant}.json','w'), ensure_ascii=False)
        pd.DataFrame(rows).to_csv('llm_results.csv', index=False)  # saved after every run
pd.DataFrame(rows).to_csv('llm_results.csv', index=False)
pd.DataFrame(rows)

In [ ]:
from google.colab import files
for f in ['transformer_coarse.csv', 'transformer_fine.csv', 'llm_results.csv']:
    if os.path.exists(f): files.download(f)